# Week 7: Content Action Playbook

**Lane:** CTR / Engagement Opportunity Scoring

This playbook turns the validated baseline queue into human-reviewed content actions. It is intentionally a prioritization aid, not an automated publishing system.

## 1) Ranked actions and reason codes

`CTR_UNDERCAPTURE` maps to `metadata_or_content_review`: inspect title, description, search intent, SERP features, and page content before changing anything. Rank is based on March CTR gap multiplied by log impression volume.

The queue is regenerated from the real March parquet partition and exported for downstream review.

In [1]:
from pathlib import Path
import duckdb
import json
import numpy as np

workspace_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'internship-warehouse').exists()), Path.cwd())
root = workspace_root / 'data' / 'internship-warehouse'
con = duckdb.connect()
march = (root / 'fact_content_daily_performance' / 'month=2026-03' / '*.parquet').as_posix()
queue = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_impressions)::DOUBLE impressions, SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions),0) ctr, SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions),0) avg_position FROM read_parquet('{march}') WHERE gsc_data_available IS TRUE GROUP BY ALL""").df()
queue['expected_ctr'] = 0.10 / queue['avg_position'].clip(lower=1).pow(0.35)
queue['ctr_gap'] = (queue['expected_ctr'] - queue['ctr']).clip(lower=0)
queue['score'] = queue['ctr_gap'] * np.log1p(queue['impressions'])
queue['reason_code'] = 'CTR_UNDERCAPTURE'
queue['action'] = 'metadata_or_content_review'
queue = queue.sort_values(['score','impressions'], ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)
output_dir = workspace_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue[['rank','client_hash_id','content_hash_id','score','reason_code','action','impressions','ctr','avg_position']].to_csv(output_dir / 'action_playbook_queue.csv', index=False)
metrics = {'lane': 'CTR / Engagement Opportunity Scoring', 'month': '2026-03', 'rows': int(len(queue)), 'top_score': float(queue.score.iloc[0]), 'reason_code': 'CTR_UNDERCAPTURE', 'action': 'metadata_or_content_review'}
(output_dir / 'action_playbook_metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Exported {len(queue):,} ranked actions')
print(queue.head(10)[['rank','score','reason_code','action']].to_string(index=False))
print(f'Wrote metrics receipt: {output_dir / "action_playbook_metrics.json"}')

Exported 176,738 ranked actions
 rank    score      reason_code                     action
    1 1.225239 CTR_UNDERCAPTURE metadata_or_content_review
    2 1.172770 CTR_UNDERCAPTURE metadata_or_content_review
    3 1.133525 CTR_UNDERCAPTURE metadata_or_content_review
    4 1.109534 CTR_UNDERCAPTURE metadata_or_content_review
    5 1.101712 CTR_UNDERCAPTURE metadata_or_content_review
    6 1.098326 CTR_UNDERCAPTURE metadata_or_content_review
    7 1.033921 CTR_UNDERCAPTURE metadata_or_content_review
    8 1.024363 CTR_UNDERCAPTURE metadata_or_content_review
    9 1.014999 CTR_UNDERCAPTURE metadata_or_content_review
   10 1.006789 CTR_UNDERCAPTURE metadata_or_content_review
Wrote metrics receipt: C:\Users\Brianna\Desktop\FLYRANK\work\outputs\action_playbook_metrics.json


## 2) Intended use and limits

**Intended use:** help an SEO or content reviewer choose which visible pages to inspect first.

**Limits:** a CTR gap is observational, not causal; pseudonymous IDs do not reveal business identity; missing GA4 coverage limits engagement interpretation; March is one feature window; and April outcomes should be used for evaluation, not decision-time ranking.

## 3) Human review and no-go list

A reviewer must check query intent, title and description accuracy, SERP features, seasonality, page quality, and whether the action is safe for the audience.

**Do not automate:** title or description publishing, content deletion, redirect changes, client prioritization, or claims that a change caused a performance increase.

## 4) Monitoring and retrain triggers

Monitor monthly queue size, impression coverage, CTR-gap distribution, missingness, reviewer acceptance rate, and future Precision@10. Recheck the rule when coverage changes materially, the CTR-gap distribution shifts, reviewer acceptance falls, or the future metric remains below the Week 4 baseline for two review cycles.

## 5) Self-check

The notebook exports ranked actions and reason codes, states intended use and limits, defines human review and no-go cases, names monitoring and retrain triggers, and writes a JSON receipt for the paper.